# Solution E3 · Normalizer is not a scaler: it normalizes the rows, not the columns

**Type**: Find the bug

## Objective

Find out why a KNN on the wines with `Normalizer` "to bring everything onto the same scale" gets 0.80 instead of 0.96: `Normalizer` divides **every row** by its length, erases the magnitude of the samples and does not touch the relative scales of the columns.
Replace it with `StandardScaler` and understand when `Normalizer` is instead the right choice.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

X, y = load_wine(return_X_y=True, as_frame=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
bug = make_pipeline(Normalizer(), KNeighborsClassifier(7))       # "that way all the columns are on the same scale"
bug_acc = cross_val_score(bug, X, y, cv=cv).mean()
print(f"accuracy with Normalizer: {bug_acc:.3f}")

**Where the bug is.** `Normalizer` is a **per-row** transformation: it divides every wine by its own length, so that the vector of the 13 measurements has norm 1. It does not remove the mean of the columns nor bring them onto the same scale. Since `proline`
(hundreds) dominates the length of every row, after the transformation `proline` is almost always 1 (its variability disappears) and what comes to dominate the distance is another column, `magnesium`, with a standard deviation more than a hundred times that of `nonflavanoid_phenols`.

In [ ]:
no_transform_acc = cross_val_score(KNeighborsClassifier(7), X, y, cv=cv).mean()
std_acc = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(7)), X, y, cv=cv).mean()
print(f"no transformation: {no_transform_acc:.3f} | Normalizer: {bug_acc:.3f} | StandardScaler: {std_acc:.3f}")

In [ ]:
Xn = Normalizer().fit_transform(X)
norms_after = np.linalg.norm(Xn, axis=1)
unit_norms = bool(np.allclose(norms_after, 1.0))
deviations_after = pd.Series(Xn.std(axis=0), index=X.columns)
dominant_column_after = deviations_after.idxmax()
scale_ratio = float(deviations_after.max() / deviations_after.min())
a, b = Normalizer().transform(np.array([X.iloc[0], 2 * X.iloc[0]]))
proportional_distance = float(np.linalg.norm(a - b))
what_normalizer_does = "c"
print("unit norms:", unit_norms, "| dominant column after Normalizer:", dominant_column_after, f"(scales: ratio {scale_ratio:.0f}x)", "| distance between x and 2x after Normalizer:", proportional_distance)
print(deviations_after.round(4).sort_values(ascending=False).head(4).to_string())

## Check

In [ ]:
with verify():
    check(bug_acc < 0.85, "the colleague gets less than 0.85", "don't modify the colleague's code")
    check(std_acc is not None and std_acc > 0.94 and std_acc > bug_acc + 0.1, "with StandardScaler the accuracy goes up by over 0.1 (above 0.94)", "make_pipeline(StandardScaler(), KNeighborsClassifier(7))")
    check(no_transform_acc is not None and bug_acc > no_transform_acc, "Normalizer improves on doing nothing, but only a little compared to the scaler: it is an accidental improvement", "no_transform_acc: just KNeighborsClassifier(7)")
    check(unit_norms is True and close(norms_after, 1.0), "after Normalizer every row has norm 1", "np.linalg.norm(..., axis=1)")
    check(dominant_column_after == deviations_after.idxmax() == "magnesium" and scale_ratio > 50, "after Normalizer another column dominates (`magnesium`) and the scales still differ by over 50 times", "std per column of the result")
    check(proportional_distance is not None and proportional_distance < 1e-12, "two wines that differ by a factor 2 have distance 0 after Normalizer", "Normalizer erases the length of the row")
    check(what_normalizer_does == "c", "correct answer: it divides every row by its norm", "c")
done()

## What you learned

- `Normalizer` **is not a scaler**: it divides every row by its norm. It is useful when the **direction** of the vector matters and not the length (word frequencies in documents of different length, histograms, cosine similarity): two documents
  equal except for a factor 2 (one long, one short) become identical. In the wines the length is information: a wine with more of everything is different from one with less of everything.
- **The columns do not end up on the same scale**: after `Normalizer` the variability of `proline` disappears (it is almost always 1) and the dominant column becomes `magnesium`, with a standard deviation more than a hundred times that of the least variable column. The result (0.80) is far below that of the scaler (0.96) and only a little better than doing nothing (0.69).
- The name is misleading: `StandardScaler` and `MinMaxScaler` transform the **columns** with training statistics; `Normalizer` transforms the **rows** without learning anything (`fit` does nothing) and the pipeline treats it like any other step.
- The warning sign was "0.80 on a dataset where a scaled KNN gets 0.96": you discover it only by comparing with **a correct baseline**. When a preprocessing "improves" a result, always ask yourself compared to what.

## Common pitfall

Choosing a transformer by its name ("normalize": so it brings everything to the same scale) without reading what it does. In scikit-learn "normalize" always means "bring the norm of the row to 1", which is different from standardizing.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is the difference between StandardScaler, MinMaxScaler and Normalizer?</summary>

The first two work per **column** with statistics learned from the training set: StandardScaler removes the mean and divides by the standard deviation, MinMaxScaler brings every column into an interval (usually 0-1). Normalizer works per **row** and divides every example by its
norm: it is suited when the direction matters (text, cosine) and does not change the relative scales of the columns. I choose based on what must be comparable: the columns with each other or the examples with each other.

</details>